# VLM 기반 한국어 대체텍스트 생성

마켓컬리 크롤링 이미지에 **Qwen2.5-VL-3B, Gemma 3 4B, HyperCLOVA X Vision 3B**를 적용해 한국어 alt를 생성하도록 구성했음.

- 이미지 파일만 모델 입력으로 사용했음
- 세 모델에 동일한 프롬프트를 사용했음
- 처음에는 이미지 5개만 테스트하도록 설정했음
- 전체 실행 시 `TEST_LIMIT = None`으로 변경하면 됨
- 결과는 `generated_alt.csv`에 저장했음


## 패키지 설치

In [ ]:
# %pip install -q "transformers==4.52.4" accelerate pandas pillow tqdm einops timm av decord

## 기본 모듈 및 설정

In [ ]:
import gc
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor

# 현재 프로젝트 폴더를 경로로 설정했음
PROJECT_ROOT = Path.cwd()

# 결과 파일 설정했음
OUTPUT_CSV = PROJECT_ROOT / "generated_alt.csv"

print("프로젝트 경로:", PROJECT_ROOT)

# 처음에는 5개만 확인하도록 설정했음
# 전체 실행 시 None으로 변경하면 됨
TEST_LIMIT = None

# 실행 모델 정리했음
RUN_MODELS = [
    "qwen",
    # "gemma", 젬마 이자식 왜 안되냐
    "hyperclovax",
]

# 모델 ID 정리했음
MODEL_IDS = {
    "qwen": "Qwen/Qwen2.5-VL-3B-Instruct",
    "gemma": "google/gemma-3-4b-it",
    "hyperclovax": "naver-hyperclovax/HyperCLOVAX-SEED-Vision-Instruct-3B",
}

# 공통 프롬프트 설정했음
PROMPT = (
    "시각장애인을 위한 웹 대체텍스트를 작성해 주세요. "
    "이미지에서 직접 확인되는 핵심 대상과 특징만 간결한 한국어 한 문장으로 설명해 주세요. "
    "보이지 않는 정보는 추측하지 마세요. "
    "'이미지에는', '사진은', '모습입니다', '있습니다' 같은 불필요한 표현은 되도록 사용하지 마세요. "
    "설명 문장만 출력해 주세요."
)

# 생성 길이 제한했음
MAX_NEW_TOKENS = 80

# 이미지 확장자 정리했음
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}


# GPU 사용 여부 확인했음
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 찾지 못했음.")

# 연산 자료형 설정했음
if torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16


print("GPU:", torch.cuda.get_device_name(0))
print("자료형:", DTYPE)
print("프로젝트:", PROJECT_ROOT)

In [ ]:
# 카테고리 폴더 확인했음
category_names = [
    "bakery",
    "convenience_meals",
    "fruits_nuts_rice",
    "vegetables",
]

for category in category_names:
    category_path = PROJECT_ROOT / category
    image_path = category_path / "images"

    print(
        category,
        "| 카테고리:", category_path.exists(),
        "| images:", image_path.exists(),
    )

## 이미지 및 기본 정보 함수

In [ ]:
# 이미지 목록 만들었음
def find_images():
    image_paths = []

    for path in PROJECT_ROOT.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        if "images" not in path.parts:
            continue

        image_paths.append(path)

    image_paths.sort()

    if TEST_LIMIT is not None:
        image_paths = image_paths[:TEST_LIMIT]

    return image_paths


# 카테고리 이름 추출했음
def get_category(image_path):
    if image_path.parent.name == "images":
        return image_path.parent.parent.name

    return image_path.parent.name


# 상품 ID 추출했음
def get_product_id(image_path):
    parts = image_path.stem.split("_")

    if len(parts) >= 3 and parts[0].lower() == "kurly":
        return parts[1]

    return ""


# 이미지 순서 추출했음
def get_image_order(image_path):
    parts = image_path.stem.split("_")

    if len(parts) >= 3 and parts[-1].isdigit():
        return parts[-1]

    return ""


# 결과 표 만들었음
def make_result_dataframe(image_paths):
    rows = []

    for image_path in image_paths:
        relative_path = image_path.relative_to(PROJECT_ROOT)

        row = {
            "category": get_category(image_path),
            "product_id": get_product_id(image_path),
            "image_order": get_image_order(image_path),
            "image_name": image_path.name,
            "image_path": str(relative_path),
        }

        rows.append(row)

    return pd.DataFrame(rows)

## 모델 로드 함수

In [ ]:
# 모델 입력 장치 찾았음
def get_model_device(model):
    for parameter in model.parameters():
        if parameter.device.type != "meta":
            return parameter.device

    return torch.device("cuda:0")


# Qwen 모델 불러왔음
def load_qwen():
    from transformers import Qwen2_5_VLForConditionalGeneration

    model_id = MODEL_IDS["qwen"]

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        device_map="auto",
    )

    processor = AutoProcessor.from_pretrained(model_id)

    model.eval()

    return model, processor


# Gemma 모델 불러왔음
def load_gemma():
    from transformers import Gemma3ForConditionalGeneration

    model_id = MODEL_IDS["gemma"]

    model = Gemma3ForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        device_map="auto",
    )

    processor = AutoProcessor.from_pretrained(model_id)

    model.eval()

    return model, processor


# HyperCLOVA X 모델 불러왔음
def load_hyperclovax():
    model_id = MODEL_IDS["hyperclovax"]

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=DTYPE,
        device_map="auto",
    )

    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    model.eval()

    return model, processor


# 모델 종류에 맞게 불러왔음
def load_model(model_name):
    if model_name == "qwen":
        return load_qwen()

    if model_name == "gemma":
        return load_gemma()

    if model_name == "hyperclovax":
        return load_hyperclovax()

    raise ValueError(f"지원하지 않는 모델임: {model_name}")

## alt 생성 함수

In [ ]:
# Qwen과 Gemma용 입력 만들었음
def generate_standard_alt(model, processor, image_path):
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": PROMPT,
                },
            ],
        }
    ]

    chat_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[chat_text],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    device = get_model_device(model)
    inputs = inputs.to(device)

    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    generated_ids = output_ids[:, input_length:]

    alt_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0]

    image.close()

    return alt_text.strip()


# HyperCLOVA X용 입력 만들었음
def generate_hyperclovax_alt(model, processor, image_path):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "filename": image_path.name,
                    "image": str(image_path),
                    "type": "image",
                },
                {
                    "text": PROMPT,
                    "type": "text",
                },
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )

    device = get_model_device(model)
    inputs = inputs.to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    # HyperCLOVA 출력 그대로 변환했음
    alt_text = processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )[0]

    return alt_text.strip()


# 모델별 alt 생성 방식 나눴음
def generate_alt(model_name, model, processor, image_path):
    if model_name == "hyperclovax":
        return generate_hyperclovax_alt(
            model,
            processor,
            image_path,
        )

    return generate_standard_alt(
        model,
        processor,
        image_path,
    )

## 실행 함수

In [ ]:
# GPU 메모리 비웠음
def clear_model(model, processor):
    del model
    del processor

    gc.collect()
    torch.cuda.empty_cache()


# 전체 이미지에 alt 생성했음
def run_model(model_name, result_df):
    print()
    print("=" * 60)
    print(f"{model_name} 시작했음")
    print("=" * 60)

    model, processor = load_model(model_name)

    column_name = f"{model_name}_alt"
    result_df[column_name] = ""

    for index in tqdm(
        range(len(result_df)),
        desc=model_name,
    ):
        relative_path = result_df.loc[index, "image_path"]
        image_path = PROJECT_ROOT / relative_path

        try:
            alt_text = generate_alt(
                model_name,
                model,
                processor,
                image_path,
            )

            result_df.loc[index, column_name] = alt_text

        except Exception as error:
            result_df.loc[index, column_name] = ""
            print()
            print(f"오류 발생했음: {image_path.name}")
            print(error)

        # 중간 결과 저장했음
        result_df.to_csv(
            OUTPUT_CSV,
            index=False,
            encoding="utf-8-sig",
        )

    clear_model(model, processor)

    print(f"{model_name} 완료했음")

    return result_df

## 실행

In [ ]:
# 실행 시작했음
image_paths = find_images()

if len(image_paths) == 0:
    raise FileNotFoundError(
        "images 폴더에서 이미지 파일을 찾지 못했음."
    )

print("사용 이미지 수:", len(image_paths))

result_df = make_result_dataframe(image_paths)

# 기본 정보 먼저 저장했음
result_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

for model_name in RUN_MODELS:
    result_df = run_model(
        model_name,
        result_df,
    )

print()
print("=" * 60)
print("전체 alt 생성 완료했음")
print("저장 경로:", OUTPUT_CSV)
print("=" * 60)

display(result_df.head())